# Setup


In [2]:
import json
import math
import re
from itertools import combinations
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
from statsmodels.stats.oneway import anova_oneway


# Constants


In [3]:
DATA_DIR = Path("/Users/casperkristiansson/Thesis Data/logs_4")

DIAGNOSTICS_DIR = Path("figures/diagnostics")
QQ_DIR = DIAGNOSTICS_DIR / "qq"
DRIFT_DIR = DIAGNOSTICS_DIR / "drift"
SUMMARY_DIR = DIAGNOSTICS_DIR

ARTIFACTS_DIR = Path("artifacts")
TABLE_DIR = ARTIFACTS_DIR / "tables"
FIGURE_DIR = ARTIFACTS_DIR / "figures"

for path in (QQ_DIR, DRIFT_DIR, SUMMARY_DIR, TABLE_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

FORMATS = ["hdf5", "zarr", "tiledb", "root"]
CODECS = ["gzip", "lz4", "zstd"]
PATTERNS = ["slice", "full"]
ROWS = [(codec, pattern) for codec in CODECS for pattern in PATTERNS]

RNG_SEED = 979969114
BOOTSTRAP_DRAWS = 10_000

RUN_RX = re.compile(r"^(?P<fmt>hdf5|zarr|tiledb|root)_(?P<codec>gzip|lz4|zstd)_(?P<pattern>slice|full)_(?P<runid>\d{8}-\d{6})\.json$")
CW_RX = re.compile(r"^(?P<fmt>hdf5|zarr|tiledb|root)_(?P<codec>gzip|lz4|zstd)_(?P<pattern>slice|full)_cloudwatch\.json$")


# Helper Functions


In [4]:
def parse_run_meta(name: str) -> dict[str, str] | None:
    match = RUN_RX.match(name)
    return match.groupdict() if match else None


def parse_cloudwatch_meta(name: str) -> dict[str, str] | None:
    match = CW_RX.match(name)
    return match.groupdict() if match else None


def load_runs(path: Path) -> pd.DataFrame:
    with open(path, "r") as handle:
        payload = json.load(handle)
    frame = pd.DataFrame(payload["data"])
    if "i" not in frame.columns:
        frame["i"] = np.arange(1, len(frame) + 1)
    frame["time_sec"] = frame["t_total_ns"] / 1e9
    frame["ln_time"] = np.log(frame["time_sec"])
    return frame


def load_latency_runs(data_dir: Path, codec: str, pattern: str) -> pd.DataFrame:
    rows = []
    for path in data_dir.iterdir():
        if not (path.is_file() and path.suffix == ".json" and "_cloudwatch" not in path.name):
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        if meta["codec"] != codec or meta["pattern"] != pattern:
            continue
        frame = load_runs(path).loc[:, ["i", "time_sec", "ln_time"]].copy()
        frame["format"] = meta["fmt"]
        frame["run_id"] = meta["runid"]
        rows.append(frame)
    runs = pd.concat(rows, ignore_index=True)
    runs = runs[runs["format"].isin(FORMATS)]
    return runs


def qq_envelope(n: int, draws: int = 2000, seed: int = RNG_SEED) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    samples = np.sort(rng.standard_normal(size=(draws, n)), axis=1)
    lower = np.percentile(samples, 2.5, axis=0)
    upper = np.percentile(samples, 97.5, axis=0)
    theoretical = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)
    return theoretical, lower, upper


def lag1_autocorr(values: np.ndarray) -> float:
    centered = values - values.mean()
    denom = np.dot(centered, centered)
    if denom == 0:
        return 0.0
    return float(np.dot(centered[:-1], centered[1:]) / denom)


def smooth_series(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    if len(x) == 0:
        return np.array([])
    try:
        return sm_lowess(y, x, frac=0.3, return_sorted=False)
    except Exception:
        window = max(3, int(round(0.3 * len(x))))
        if window % 2 == 0:
            window += 1
        pad = window // 2
        padded = np.pad(y, (pad, pad), mode="edge")
        kernel = np.ones(window) / window
        return np.convolve(padded, kernel, mode="valid")


def nice_ylim(ymin: float, ymax: float, pad_frac: float = 0.1, floor: float | None = None) -> tuple[float, float]:
    if not np.isfinite(ymin) or not np.isfinite(ymax):
        base = floor if floor is not None else 0.0
        return base, base + 1.0
    if ymax <= ymin:
        ymax = ymin + 1.0
    span = ymax - ymin
    pad = pad_frac * span if span > 0 else 1.0
    low = ymin - pad
    high = ymax + pad
    if floor is not None:
        low = max(floor, low)
    return low, high


def garwood_rate_ci(k: float, exposure: float, alpha: float = 0.05) -> tuple[float, float]:
    if not np.isfinite(exposure) or exposure <= 0 or not np.isfinite(k) or k < 0:
        return np.nan, np.nan
    lower = 0.0 if k == 0 else 0.5 * stats.chi2.ppf(alpha / 2, 2 * k)
    upper = 0.5 * stats.chi2.ppf(1 - alpha / 2, 2 * (k + 1))
    return lower / exposure, upper / exposure


def gmean_ci_from_ln(values: np.ndarray, alpha: float = 0.05) -> tuple[float, float, float]:
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    mean = float(values.mean())
    std = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    if len(values) > 1 and np.isfinite(std) and std > 0:
        tcrit = stats.t.ppf(1 - alpha / 2, len(values) - 1)
        delta = tcrit * std / math.sqrt(len(values))
        return math.exp(mean), math.exp(mean - delta), math.exp(mean + delta)
    val = math.exp(mean)
    return val, val, val


def bootstrap_quantile(values: np.ndarray, q: float, draws: int, seed: int) -> tuple[float, float, float]:
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(draws, len(values)), replace=True)
    estimates = np.quantile(samples, q, axis=1)
    point = float(np.quantile(values, q))
    lo, hi = np.quantile(estimates, [0.025, 0.975])
    return point, float(lo), float(hi)


def welch_anova_ln(groups: dict[str, np.ndarray]) -> dict[str, float]:
    arrays = [np.asarray(v, dtype=float) for v in groups.values() if len(v) > 0]
    if len(arrays) < 2:
        return {"F": np.nan, "df_between": np.nan, "df_within": np.nan, "pvalue": np.nan}
    res = anova_oneway(arrays, use_var="unequal", welch_correction=True)
    return {
        "F": float(res.statistic),
        "df_between": float(res.df_num),
        "df_within": float(res.df_denom),
        "pvalue": float(res.pvalue),
    }


def omega_sq_oneway_ln(groups: dict[str, np.ndarray]) -> float:
    arrays = [np.asarray(v, dtype=float) for v in groups.values() if len(v) > 0]
    if len(arrays) < 2:
        return float("nan")
    means = [float(arr.mean()) for arr in arrays]
    ns = [len(arr) for arr in arrays]
    all_values = np.concatenate(arrays)
    grand = float(all_values.mean())
    ss_within = sum(np.sum((arr - mean) ** 2) for arr, mean in zip(arrays, means))
    ss_between = sum(n * (mean - grand) ** 2 for n, mean in zip(ns, means))
    sst = ss_within + ss_between
    k = len(arrays)
    total_n = sum(ns)
    if total_n <= k:
        return float("nan")
    ms_within = ss_within / (total_n - k)
    if not np.isfinite(ms_within):
        return float("nan")
    denom = sst + ms_within
    if denom == 0:
        return float("nan")
    return float((ss_between - (k - 1) * ms_within) / denom)


def holm_adjust(pvals: pd.Series) -> pd.Series:
    if pvals.size == 0:
        return pvals
    order = pvals.sort_values().index.tolist()
    adjusted: dict[tuple[str, str], float] = {}
    m = len(order)
    for position, key in enumerate(order):
        factor = m - position
        adjusted[key] = min(1.0, factor * float(pvals.loc[key]))
    for idx in range(1, len(order)):
        prev = order[idx - 1]
        curr = order[idx]
        adjusted[curr] = max(adjusted[curr], adjusted[prev])
    return pd.Series(adjusted)[pvals.index]



def games_howell_ln(frame: pd.DataFrame, group_col: str, value_col: str) -> pd.DataFrame:
    result = pg.pairwise_gameshowell(dv=value_col, between=group_col, data=frame).reset_index(drop=True)

    ci_lower = None
    ci_upper = None
    ci_candidates = [col for col in result.columns if 'CI95' in col.replace(' ', '').upper()]

    def _series_as_tuple(series: pd.Series) -> tuple[pd.Series, pd.Series] | None:
        try:
            tuples = series.apply(lambda v: (float(v[0]), float(v[1])))
        except Exception:
            return None
        lo = tuples.map(lambda pair: pair[0])
        hi = tuples.map(lambda pair: pair[1])
        return lo, hi

    for column in ci_candidates:
        label = column.lower()
        series = result[column]
        if any(token in label for token in ['_low', '_lower', '-']) or label.endswith('lo'):
            ci_lower = series.astype(float)
        elif any(token in label for token in ['_high', '_upper', '+']) or label.endswith('hi'):
            ci_upper = series.astype(float)

    if (ci_lower is None or ci_upper is None) and 'CI95%' in result.columns:
        parsed = _series_as_tuple(result['CI95%'])
        if parsed is not None:
            ci_lower, ci_upper = parsed

    if (ci_lower is None or ci_upper is None) and ci_candidates:
        fallback = _series_as_tuple(result[ci_candidates[0]])
        if fallback is not None:
            ci_lower, ci_upper = fallback

    if ci_lower is None or ci_upper is None:
        se_col = next((col for col in ['se', 'SE'] if col in result.columns), None)
        df_col = next((col for col in ['dof', 'df'] if col in result.columns), None)
        if se_col and df_col:
            diff = (result['mean(A)'] - result['mean(B)']).astype(float)
            se = result[se_col].astype(float)
            dof = result[df_col].astype(float)
            tcrit = dof.apply(lambda df: stats.t.ppf(0.975, df) if np.isfinite(df) and df > 0 else np.nan)
            ci_lower = diff - tcrit * se
            ci_upper = diff + tcrit * se
        else:
            raise KeyError('Unable to determine CI95% bounds from pingouin.pairwise_gameshowell output')

    diff = (result['mean(A)'] - result['mean(B)']).astype(float)
    out = pd.DataFrame({
        'i': result['A'].astype(str),
        'j': result['B'].astype(str),
        'diff_ln': diff,
        'ratio': np.exp(diff),
        'ratio_ci_lo': np.exp(ci_lower.astype(float)),
        'ratio_ci_hi': np.exp(ci_upper.astype(float)),
        'p_raw': result['pval'].astype(float),
    })
    indexed = out.set_index(['i', 'j'])
    indexed['p_holm'] = holm_adjust(indexed['p_raw']).astype(float)
    return indexed.reset_index()

def compact_letter_display(pairs: pd.DataFrame, alpha: float = 0.05) -> dict[str, str]:
    items = sorted(set(pairs["i"]).union(set(pairs["j"])))
    letters = {item: "" for item in items}
    significant = {(row.i, row.j) for row in pairs.itertuples(index=False) if row.p_holm < alpha}
    significant |= {(b, a) for (a, b) in significant}
    remaining = set(items)
    current = ord("A")
    while remaining:
        group = []
        for item in sorted(remaining):
            if all((item, other) not in significant for other in group):
                group.append(item)
        for item in group:
            letters[item] += chr(current)
        remaining -= set(group)
        current += 1
    return letters


# Diagnostic Summaries


In [5]:
def plot_qq_ln_times(frame: pd.DataFrame, meta: dict[str, str], outdir: Path) -> None:
    z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
    z_sorted = np.sort(z.values)
    n = len(z_sorted)
    theoretical, envelope_lo, envelope_hi = qq_envelope(n)
    statistic, p_value = stats.shapiro(frame["ln_time"].values)
    plt.figure(figsize=(5.5, 5.5))
    plt.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.2, label="95% envelope")
    plt.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=3, label="sample")
    limit = np.nanmax(np.abs(np.concatenate([theoretical, z_sorted])))
    limit = float(np.ceil(limit * 1.05 * 10) / 10)
    plt.plot([-limit, limit], [-limit, limit], linestyle="--", linewidth=1)
    title = f"Q–Q ln(time): {meta['fmt']}/{meta['codec']}/{meta['pattern']} | n={n} | Shapiro p={p_value:.3g}"
    plt.title(title)
    plt.xlabel("Theoretical z (N(0,1))")
    plt.ylabel("Sample z (standardized ln time)")
    plt.tight_layout()
    filename = f"qq_{meta['fmt']}_{meta['codec']}_{meta['pattern']}.png"
    plt.savefig(outdir / filename, dpi=150)
    plt.close()


def plot_drift_ln_times(frame: pd.DataFrame, meta: dict[str, str], outdir: Path) -> None:
    x = frame["i"].values
    y = frame["ln_time"].values
    geometric = float(np.exp(y.mean()))
    rho1 = lag1_autocorr(y)
    plt.figure(figsize=(7, 4))
    plt.plot(x, y, marker="o", linestyle="-", linewidth=1)
    y_smooth = smooth_series(x, y)
    plt.plot(x, y_smooth, linewidth=2)
    plt.axhline(np.log(geometric), linestyle="--", linewidth=1)
    title = f"Run-order ln(time): {meta['fmt']}/{meta['codec']}/{meta['pattern']} | n={len(x)} | ρ₁={rho1:.2f}"
    plt.title(title)
    plt.xlabel("Repetition index")
    plt.ylabel("ln(time [s])")
    plt.tight_layout()
    filename = f"drift_{meta['fmt']}_{meta['codec']}_{meta['pattern']}.png"
    plt.savefig(outdir / filename, dpi=150)
    plt.close()


def build_run_summary(data_dir: Path, qq_dir: Path, drift_dir: Path) -> pd.DataFrame:
    fields = [
        "file",
        "format",
        "codec",
        "pattern",
        "run_id",
        "n",
        "geometric_mean_sec",
        "p95_sec",
        "shapiro_W",
        "shapiro_p",
        "rho1",
        "spearman_rho",
        "spearman_p",
    ]
    summaries: list[dict[str, object]] = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file() or not path.name.endswith(".json") or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        frame = load_runs(path)
        plot_qq_ln_times(frame, meta, qq_dir)
        plot_drift_ln_times(frame, meta, drift_dir)
        gm = float(np.exp(frame["ln_time"].mean()))
        p95 = float(np.exp(np.quantile(frame["ln_time"], 0.95)))
        shapiro_stat, shapiro_p = stats.shapiro(frame["ln_time"].values)
        rho1 = lag1_autocorr(frame["ln_time"].values)
        rho_s, p_s = stats.spearmanr(frame["i"].values, frame["ln_time"].values)
        summaries.append({
            "file": path.name,
            "format": meta["fmt"],
            "codec": meta["codec"],
            "pattern": meta["pattern"],
            "run_id": meta["runid"],
            "n": int(len(frame)),
            "geometric_mean_sec": gm,
            "p95_sec": p95,
            "shapiro_W": float(shapiro_stat),
            "shapiro_p": float(shapiro_p),
            "rho1": float(rho1),
            "spearman_rho": float(rho_s) if np.isfinite(rho_s) else np.nan,
            "spearman_p": float(p_s) if np.isfinite(p_s) else np.nan,
        })
    if not summaries:
        raise ValueError("No diagnostic summaries were produced.")
    summary = pd.DataFrame(summaries, columns=fields)
    summary = summary.sort_values(["format", "codec", "pattern", "run_id"]).reset_index(drop=True)
    summary["p95_over_gm"] = summary["p95_sec"] / summary["geometric_mean_sec"]
    summary["non_normal_flag"] = summary["shapiro_p"] < 0.05
    return summary


def format_run_summary_table(summary: pd.DataFrame) -> pd.DataFrame:
    if len(summary) == 0:
        raise ValueError("Summary table is empty.")
    table = summary.loc[:, [
        "format",
        "codec",
        "pattern",
        "n",
        "shapiro_p",
        "non_normal_flag",
        "rho1",
        "p95_over_gm",
        "spearman_p",
    ]].copy()
    table = table.rename(columns={
        "format": "Format",
        "codec": "Codec",
        "pattern": "Pattern",
        "n": "n",
        "shapiro_p": r"Shapiro $p$",
        "non_normal_flag": r"Non-normal (p<0.05)",
        "rho1": r"$
ho_1$",
        "p95_over_gm": r"$\mathrm{p95}/\mathrm{GM}$",
        "spearman_p": r"Spearman $p$",
    })
    table[r"Shapiro $p$"] = table[r"Shapiro $p$"].map(lambda x: f"{x:.3g}")
    table[r"Non-normal (p<0.05)"] = table[r"Non-normal (p<0.05)"].map(lambda v: r"	extbf{Yes}" if v else "No")
    table[r"$
ho_1$"] = table[r"$
ho_1$"].map(lambda x: f"{x:.2f}")
    table[r"$\mathrm{p95}/\mathrm{GM}$"] = table[r"$\mathrm{p95}/\mathrm{GM}$"].map(lambda x: f"{x:.2f}")
    table[r"Spearman $p$"] = table[r"Spearman $p$"].map(lambda x: "nan" if pd.isna(x) else f"{x:.3g}")
    table = table.sort_values(["Codec", "Pattern", "Format"]).reset_index(drop=True)
    return table


SyntaxError: unterminated string literal (detected at line 119) (1547956809.py, line 119)

In [ ]:
run_summary = build_run_summary(DATA_DIR, QQ_DIR, DRIFT_DIR)
if len(run_summary) == 0:
    raise ValueError("Diagnostic summary is empty.")
run_summary.to_csv(SUMMARY_DIR / "diagnostics_summary_full.csv", index=False)
run_summary_table = format_run_summary_table(run_summary)
run_summary_table.to_csv(SUMMARY_DIR / "diagnostics_summary_table.csv", index=False)


# Appendix Grids


In [ ]:
def latest_runs(data_dir: Path) -> dict[tuple[str, str, str], tuple[str, Path]]:
    latest: dict[tuple[str, str, str], tuple[str, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or not path.name.endswith(".json") or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        run_id = meta["runid"]
        record = latest.get(key)
        if record is None or run_id > record[0]:
            latest[key] = (run_id, path)
    return latest


def generate_appendix_grids(data_dir: Path, dest_dir: Path) -> None:
    latest = latest_runs(data_dir)
    cells_data: dict[tuple[str, str, str], dict[str, object]] = {}
    for fmt in FORMATS:
        for codec, pattern in ROWS:
            key = (fmt, codec, pattern)
            if key not in latest:
                continue
            run_id, path = latest[key]
            frame = load_runs(path)
            cells_data[key] = {
                "frame": frame,
                "run_id": run_id,
                "shapiro_p": stats.shapiro(frame["ln_time"].values)[1],
                "rho1": lag1_autocorr(frame["ln_time"].values),
            }
    if not cells_data:
        return
    fig, axes = plt.subplots(len(ROWS), len(FORMATS), figsize=(14, 18), sharex=True, sharey=True)
    for r, (codec, pattern) in enumerate(ROWS):
        for c, fmt in enumerate(FORMATS):
            ax = axes[r, c]
            key = (fmt, codec, pattern)
            if key not in cells_data:
                ax.axis("off")
                continue
            frame = cells_data[key]["frame"]
            z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
            z_sorted = np.sort(z.values)
            theoretical, envelope_lo, envelope_hi = qq_envelope(len(z_sorted))
            ax.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.15)
            ax.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=2)
            lim = 3.0
            ax.plot([-lim, lim], [-lim, lim], linestyle="--", linewidth=0.8)
            ax.set_xlim(-lim, lim)
            ax.set_ylim(-lim, lim)
            if r == 0:
                ax.set_title(fmt, fontsize=10)
            if c == 0:
                ax.set_ylabel(f"{codec}/{pattern}", fontsize=9)
            ax.text(0.98, 0.02, f"p={cells_data[key]['shapiro_p']:.3g}", transform=ax.transAxes, ha="right", va="bottom", fontsize=8)
    fig.suptitle("Q–Q plots of ln(time) by codec/pattern and format", fontsize=12)
    fig.text(0.5, 0.005, "Theoretical z (N(0,1))", ha="center")
    fig.text(0.005, 0.5, "Sample z (standardized ln time)", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(dest_dir / "appendix_QQ_grid_6x4.png", dpi=300)
    plt.savefig(dest_dir / "appendix_QQ_grid_6x4.pdf")
    plt.close(fig)
    fig, axes = plt.subplots(len(ROWS), len(FORMATS), figsize=(14, 18), sharex=True, sharey=False)
    for r, (codec, pattern) in enumerate(ROWS):
        for c, fmt in enumerate(FORMATS):
            ax = axes[r, c]
            key = (fmt, codec, pattern)
            if key not in cells_data:
                ax.axis("off")
                continue
            frame = cells_data[key]["frame"]
            x = frame["i"].values
            y = frame["ln_time"].values
            quantiles = np.quantile(y, [0.02, 0.98])
            if not np.isfinite(quantiles).all() or quantiles[1] <= quantiles[0]:
                ymin, ymax = float(y.min()), float(y.max())
            else:
                ymin, ymax = float(quantiles[0]), float(quantiles[1])
            span = max(ymax - ymin, 1e-3)
            pad = 0.05 * span
            ax.set_ylim(ymin - pad, ymax + pad)
            ax.plot(x, y, marker="o", linestyle="-", linewidth=0.8, markersize=2)
            y_smooth = smooth_series(x, y)
            ax.plot(x, y_smooth, linewidth=1.5)
            geometric = float(np.exp(y.mean()))
            ax.axhline(np.log(geometric), linestyle="--", linewidth=0.8)
            if r == 0:
                ax.set_title(fmt, fontsize=10)
            if c == 0:
                ax.set_ylabel(f"{codec}/{pattern}", fontsize=9)
            if r == len(ROWS) - 1:
                ax.set_xlabel("rep", fontsize=9)
            ax.text(0.98, 0.02, f"ρ₁={cells_data[key]['rho1']:.2f}", transform=ax.transAxes, ha="right", va="bottom", fontsize=8)
    fig.suptitle("Run-order traces of ln(time) by codec/pattern and format", fontsize=12)
    fig.text(0.005, 0.5, "ln(time [s])", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(dest_dir / "appendix_Drift_grid_6x4.png", dpi=300)
    plt.savefig(dest_dir / "appendix_Drift_grid_6x4.pdf")
    plt.close(fig)


In [ ]:
generate_appendix_grids(DATA_DIR, DIAGNOSTICS_DIR)


# Request Intensity


In [ ]:
def latest_run_by_mtime(data_dir: Path) -> dict[tuple[str, str, str], Path]:
    latest: dict[tuple[str, str, str], tuple[float, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        record = latest.get(key)
        mtime = path.stat().st_mtime
        if record is None or mtime > record[0]:
            latest[key] = (mtime, path)
    return {key: value for key, (_, value) in latest.items()}


def latest_cloudwatch_by_mtime(data_dir: Path) -> dict[tuple[str, str, str], Path]:
    latest: dict[tuple[str, str, str], tuple[float, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "_cloudwatch" not in path.name:
            continue
        meta = parse_cloudwatch_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        record = latest.get(key)
        mtime = path.stat().st_mtime
        if record is None or mtime > record[0]:
            latest[key] = (mtime, path)
    return {key: value for key, (_, value) in latest.items()}


def compute_request_intensity_table(data_dir: Path) -> pd.DataFrame:
    run_files = latest_run_by_mtime(data_dir)
    cloudwatch_files = latest_cloudwatch_by_mtime(data_dir)
    rows: list[dict[str, object]] = []
    for key in sorted(set(run_files) & set(cloudwatch_files)):
        fmt, codec, pattern = key
        with open(run_files[key], "r") as handle:
            run_payload = json.load(handle)
        n_included = int(len(run_payload["data"]))
        with open(cloudwatch_files[key], "r") as handle:
            cw_payload = json.load(handle)
        gets = float(cw_payload.get("GetRequestsSum", np.nan))
        bytes_downloaded = float(cw_payload.get("BytesDownloadedSum", np.nan))
        gib = bytes_downloaded / (2 ** 30) if np.isfinite(bytes_downloaded) else np.nan
        gets_per_rep = gets / n_included if n_included > 0 else np.nan
        gets_per_gib = gets / gib if np.isfinite(gib) and gib > 0 else np.nan
        rep_lo, rep_hi = garwood_rate_ci(gets, n_included)
        gib_lo, gib_hi = garwood_rate_ci(gets, gib)
        rows.append({
            "format": fmt,
            "codec": codec,
            "pattern": pattern,
            "GetRequestsSum": gets,
            "BytesDownloadedSum_GiB": gib,
            "n_included": n_included,
            "GETs_per_rep": gets_per_rep,
            "GETs_per_rep_lo": rep_lo,
            "GETs_per_rep_hi": rep_hi,
            "GETs_per_GiB": gets_per_gib,
            "GETs_per_GiB_lo": gib_lo,
            "GETs_per_GiB_hi": gib_hi,
            "run_file": run_files[key].name,
            "cw_file": cloudwatch_files[key].name,
        })
    frame = pd.DataFrame(rows)
    frame = frame.sort_values(["codec", "pattern", "format"]).reset_index(drop=True)
    return frame


def plot_request_intensity_series(frame: pd.DataFrame, metric: str, lower: str, upper: str, prefix: str, ylabel: str) -> None:
    for codec in CODECS:
        for pattern in PATTERNS:
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].copy()
            subset = subset.set_index("format").reindex(FORMATS).dropna(subset=[metric]).reset_index()
            x = np.arange(len(subset))
            values = subset[metric].to_numpy(float)
            lower_bounds = np.maximum(0.0, subset[lower].to_numpy(float))
            upper_bounds = np.maximum(0.0, subset[upper].to_numpy(float))
            errors = [np.maximum(0.0, values - lower_bounds), np.maximum(0.0, upper_bounds - values)]
            ymin = float(np.nanmin(np.minimum(lower_bounds, values)))
            ymax = float(np.nanmax(np.maximum(upper_bounds, values)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.08, floor=0.0)
            plt.figure(figsize=(6.0, 3.2))
            ax = plt.gca()
            ax.errorbar(x, values, yerr=errors, fmt="o", capsize=4, linewidth=1)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            ax.set_ylim(lo, hi)
            ax.set_ylabel(ylabel)
            ax.set_title(f"Request intensity — {codec}/{pattern}")
            ax.yaxis.grid(True, alpha=0.3)
            for xi, yi in zip(x, values):
                ax.text(xi, yi, f"{yi:.3g}", ha="center", va="bottom", fontsize=8)
            plt.tight_layout()
            plt.savefig(FIGURE_DIR / f"{prefix}_{codec}_{pattern}.png", dpi=180)
            plt.close()


def plot_request_intensity_grid(frame: pd.DataFrame, metric: str, lower: str, upper: str, ylabel: str, prefix: str, minimum_ci_fraction: float = 0.003) -> None:
    fig, axes = plt.subplots(nrows=len(CODECS), ncols=len(PATTERNS), figsize=(9.5, 9.0), sharex=False, sharey=False)
    for r, codec in enumerate(CODECS):
        for c, pattern in enumerate(PATTERNS):
            ax = axes[r, c] if len(CODECS) > 1 else axes[c]
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].set_index("format").reindex(FORMATS).reset_index()
            subset = subset.dropna(subset=[metric])
            ax.set_title(f"{codec} / {pattern}", fontsize=10)
            x = np.arange(len(subset))
            values = subset[metric].to_numpy(float)
            lower_bounds = np.maximum(0.0, subset[lower].to_numpy(float))
            upper_bounds = np.maximum(0.0, subset[upper].to_numpy(float))
            err_low_true = np.maximum(0.0, values - lower_bounds)
            err_high_true = np.maximum(0.0, upper_bounds - values)
            epsilon = np.maximum(minimum_ci_fraction * np.maximum(values, 1.0), 1e-9)
            err_low = np.maximum(err_low_true, epsilon)
            err_high = np.maximum(err_high_true, epsilon)
            text_positions = [yi + err_hi + 0.01 * np.maximum(yi, 1.0) for yi, err_hi in zip(values, err_high)]
            ymin = float(np.nanmin(np.minimum(lower_bounds, values)))
            ymax = float(np.nanmax(np.maximum(upper_bounds, values)))
            if text_positions:
                ymax = max(ymax, float(np.nanmax(text_positions)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.15, floor=0.0)
            ax.errorbar(x, values, yerr=[err_low, err_high], fmt="o", markersize=4, markeredgewidth=0.8, elinewidth=1.6, capsize=5)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            ax.set_ylim(lo, hi)
            ax.yaxis.grid(True, alpha=0.35)
            if c == 0:
                ax.set_ylabel(ylabel)
            for xi, yi, text_y in zip(x, values, text_positions):
                clipped = min(text_y, hi - 0.02 * (hi - lo))
                if xi == 0 and len(x) > 1:
                    align = "left"
                elif xi == len(x) - 1 and len(x) > 1:
                    align = "right"
                else:
                    align = "center"
                ax.text(xi, clipped, f"{yi:.3g}", ha=align, va="bottom", fontsize=8, clip_on=True)
    fig.suptitle(ylabel, fontsize=12)
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf")
    plt.close(fig)


In [ ]:
request_intensity = compute_request_intensity_table(DATA_DIR)
if len(request_intensity) == 0:
    raise ValueError("Request intensity table is empty.")
request_intensity.to_csv(TABLE_DIR / "tab_5_4_request_intensity.csv", index=False)
plot_request_intensity_series(
    request_intensity,
    metric="GETs_per_rep",
    lower="GETs_per_rep_lo",
    upper="GETs_per_rep_hi",
    prefix="fig_5_4_reqintensity_perRep_point_ci",
    ylabel="GETs per repetition",
)
plot_request_intensity_series(
    request_intensity,
    metric="GETs_per_GiB",
    lower="GETs_per_GiB_lo",
    upper="GETs_per_GiB_hi",
    prefix="fig_5_4_reqintensity_perGiB_point_ci",
    ylabel="GETs per GiB",
)
plot_request_intensity_grid(
    request_intensity,
    metric="GETs_per_rep",
    lower="GETs_per_rep_lo",
    upper="GETs_per_rep_hi",
    ylabel="GETs per repetition",
    prefix="fig_5_4_reqintensity_perRep_grid",
    minimum_ci_fraction=0.003,
)
plot_request_intensity_grid(
    request_intensity,
    metric="GETs_per_GiB",
    lower="GETs_per_GiB_lo",
    upper="GETs_per_GiB_hi",
    ylabel="GETs per GiB",
    prefix="fig_5_4_reqintensity_perGiB_grid",
    minimum_ci_fraction=0.003,
)


# CloudWatch Latencies


In [ ]:
def aggregate_cloudwatch_latencies(data_dir: Path) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "_cloudwatch" not in path.name:
            continue
        meta = parse_cloudwatch_meta(path.name)
        if not meta:
            continue
        with open(path, "r") as handle:
            payload = json.load(handle)
        records.append({
            "format": meta["fmt"],
            "codec": meta["codec"],
            "pattern": meta["pattern"],
            "GetRequestsSum": float(payload.get("GetRequestsSum", np.nan)),
            "BytesDownloadedSum": float(payload.get("BytesDownloadedSum", np.nan)),
            "FirstByteLatencyAverage": float(payload.get("FirstByteLatencyAverage", np.nan)),
            "FirstByteLatencyP95": float(payload.get("FirstByteLatencyP95", np.nan)),
            "TotalRequestLatencyAverage": float(payload.get("TotalRequestLatencyAverage", np.nan)),
            "TotalRequestLatencyP95": float(payload.get("TotalRequestLatencyP95", np.nan)),
            "source": path.name,
        })
    sessions = pd.DataFrame(records)
    if len(sessions) == 0:
        raise ValueError("CloudWatch latency sessions are empty.")
    def reducer(group: pd.DataFrame) -> pd.Series:
        weights = group["GetRequestsSum"].to_numpy(float)
        weighted_sum = float(np.nansum(weights)) if np.isfinite(np.nansum(weights)) else 0.0
        def weighted_average(series: pd.Series) -> float:
            if weighted_sum > 0 and np.isfinite(series).all():
                return float(np.average(series.to_numpy(float), weights=weights))
            return float(series.mean())
        return pd.Series({
            "sessions": int(len(group)),
            "GetRequestsSum_total": float(group["GetRequestsSum"].sum()),
            "BytesDownloadedSum_total": float(group["BytesDownloadedSum"].sum()),
            "FirstByteLatencyAverage_ms": weighted_average(group["FirstByteLatencyAverage"]),
            "FirstByteLatencyP95_ms": float(group["FirstByteLatencyP95"].mean()),
            "TotalRequestLatencyAverage_ms": weighted_average(group["TotalRequestLatencyAverage"]),
            "TotalRequestLatencyP95_ms": float(group["TotalRequestLatencyP95"].mean()),
        })
    aggregated = sessions.groupby(["format", "codec", "pattern"]).apply(reducer).reset_index()
    aggregated = aggregated.sort_values(["codec", "pattern", "format"]).reset_index(drop=True)
    rounded = aggregated.copy()
    for column in [
        "FirstByteLatencyAverage_ms",
        "FirstByteLatencyP95_ms",
        "TotalRequestLatencyAverage_ms",
        "TotalRequestLatencyP95_ms",
    ]:
        rounded[column] = rounded[column].map(lambda x: np.nan if pd.isna(x) else round(float(x), 3))
    rounded.to_csv(TABLE_DIR / "tab_5_5_cw_latencies.csv", index=False)
    return aggregated


def plot_latency_grid(frame: pd.DataFrame, average_col: str, p95_col: str, title: str, prefix: str, ylabel: str) -> None:
    fig, axes = plt.subplots(nrows=len(CODECS), ncols=len(PATTERNS), figsize=(9.5, 9.0), sharex=False, sharey=False)
    width = 0.35
    for r, codec in enumerate(CODECS):
        for c, pattern in enumerate(PATTERNS):
            ax = axes[r, c] if len(CODECS) > 1 else axes[c]
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].set_index("format").reindex(FORMATS).reset_index()
            subset = subset.dropna(subset=[average_col, p95_col])
            if len(subset) == 0:
                raise ValueError(f"Missing latency data for {codec}/{pattern}.")
            ax.set_title(f"{codec} / {pattern}", fontsize=10)
            x = np.arange(len(subset))
            avg_vals = subset[average_col].to_numpy(float)
            p95_vals = subset[p95_col].to_numpy(float)
            ax.bar(x - width / 2, avg_vals, width, label="avg")
            ax.bar(x + width / 2, p95_vals, width, label="p95")
            ymin = float(np.nanmin(np.minimum(avg_vals, p95_vals)))
            ymax = float(np.nanmax(np.maximum(avg_vals, p95_vals)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.12, floor=0.0)
            ax.set_ylim(lo, hi)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            if c == 0:
                ax.set_ylabel(ylabel)
            ax.yaxis.grid(True, alpha=0.35)
            for xi, yi in zip(x - width / 2, avg_vals):
                ax.text(xi, yi, f"{yi:.1f}", ha="center", va="bottom", fontsize=8)
            for xi, yi in zip(x + width / 2, p95_vals):
                ax.text(xi, yi, f"{yi:.1f}", ha="center", va="bottom", fontsize=8)
            if r == 0 and c == len(PATTERNS) - 1:
                ax.legend(loc="upper right", fontsize=9)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf")
    plt.close(fig)


In [ ]:
cw_aggregated = aggregate_cloudwatch_latencies(DATA_DIR)
if len(cw_aggregated) == 0:
    raise ValueError("CloudWatch aggregate table is empty.")
plot_latency_grid(
    cw_aggregated,
    average_col="FirstByteLatencyAverage_ms",
    p95_col="FirstByteLatencyP95_ms",
    title="S3 FirstByteLatency: avg vs p95",
    prefix="fig_5_5_firstbyte_grid",
    ylabel="FirstByte latency (ms)",
)
plot_latency_grid(
    cw_aggregated,
    average_col="TotalRequestLatencyAverage_ms",
    p95_col="TotalRequestLatencyP95_ms",
    title="S3 TotalRequestLatency: avg vs p95",
    prefix="fig_5_5_total_grid",
    ylabel="TotalRequest latency (ms)",
)


# Latency Analysis


In [19]:
def analyze_latency(codec: str, pattern: str) -> None:
    runs = load_latency_runs(DATA_DIR, codec, pattern)
    order_candidates = runs["format"].unique()
    if len(order_candidates) == 0:
        raise ValueError(f"No runs available for {codec}/{pattern}.")

    token = f"{pattern}_{codec}"
    base_seed = RNG_SEED + 97 * CODECS.index(codec) + 13 * PATTERNS.index(pattern)

    summary_rows = []
    for fmt, group in runs.groupby("format"):
        ln_values = group["ln_time"].to_numpy()
        latency_ms = 1000.0 * group["time_sec"].to_numpy()
        gm, gm_lo, gm_hi = gmean_ci_from_ln(ln_values)
        p95, p95_lo, p95_hi = bootstrap_quantile(
            latency_ms,
            q=0.95,
            draws=BOOTSTRAP_DRAWS,
            seed=base_seed + FORMATS.index(fmt),
        )
        summary_rows.append({
            "format": fmt,
            "n": len(group),
            "gm_ms": 1000.0 * gm,
            "gm_lo_ms": 1000.0 * gm_lo,
            "gm_hi_ms": 1000.0 * gm_hi,
            "p95_ms": p95,
            "p95_lo_ms": p95_lo,
            "p95_hi_ms": p95_hi,
        })

    summary_df = pd.DataFrame(summary_rows)
    sorted_by_mean = summary_df.sort_values("gm_ms", ascending=True).reset_index(drop=True)
    order = sorted_by_mean["format"].tolist()

    summary_df.sort_values(["format"]).to_csv(
        TABLE_DIR / f"tab_5_6a_{token}_latency.csv",
        index=False,
    )

    x_positions = np.arange(len(sorted_by_mean))
    gm_values = sorted_by_mean["gm_ms"].to_numpy(float)
    ci_lo = sorted_by_mean["gm_lo_ms"].to_numpy(float)
    ci_hi = sorted_by_mean["gm_hi_ms"].to_numpy(float)
    yerr = np.vstack([gm_values - ci_lo, ci_hi - gm_values])

    plt.figure(figsize=(6.2, 3.6))
    plt.errorbar(
        x_positions,
        gm_values,
        yerr=yerr,
        fmt="o",
        capsize=5,
        elinewidth=1.6,
    )
    plt.xticks(x_positions, sorted_by_mean["format"])
    plt.ylabel("Geometric mean latency (ms)")
    plt.title(f"{pattern} latency — geometric mean ±95% CI, {codec}")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_5_6a_{token}_mean_ci_ms.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"fig_5_6a_{token}_mean_ci_ms.pdf")
    plt.close()

    runs_centered = runs.copy()
    runs_centered["ln_resid"] = (
        runs_centered["ln_time"]
        - runs_centered.groupby("format")["ln_time"].transform("mean")
    )

    plt.figure(figsize=(6.2, 3.6))
    data_resid = [
        runs_centered.loc[runs_centered["format"] == fmt, "ln_resid"].to_numpy()
        for fmt in order
    ]
    plt.violinplot(data_resid, showmeans=False, showextrema=False, showmedians=False)
    rng = np.random.default_rng(base_seed)
    for idx, fmt in enumerate(order, start=1):
        points = runs_centered.loc[runs_centered["format"] == fmt, "ln_resid"].to_numpy()
        jitter = rng.uniform(-0.08, 0.08, size=len(points))
        plt.plot(idx + jitter, points, "o", markersize=2.2, alpha=0.6)
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.ylim(-0.25, 0.25)
    plt.xticks(range(1, len(order) + 1), order)
    plt.ylabel("Centered ln(time) [s] (ln t − group mean)")
    plt.title(f"{pattern} latency — centered dispersion, {codec}")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_5_6b_{token}_centered_violin_ln.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"fig_5_6b_{token}_centered_violin_ln.pdf")
    plt.close()

    plt.figure(figsize=(6.4, 3.8))
    for fmt in order:
        series = 1000.0 * runs.loc[runs["format"] == fmt, "time_sec"].to_numpy()
        x = np.sort(series)
        y = np.arange(1, len(x) + 1) / len(x)
        plt.plot(x, y, label=fmt)
    plt.xlabel("Latency (ms)")
    plt.ylabel("ECDF")
    plt.title(f"{pattern} latency — ECDF, {codec}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_5_6c_{token}_ecdf_ms.png", dpi=200)
    plt.savefig(FIGURE_DIR / f"fig_5_6c_{token}_ecdf_ms.pdf")
    plt.close()

    groups = {fmt: runs.loc[runs["format"] == fmt, "ln_time"].to_numpy() for fmt in order}
    anova = welch_anova_ln(groups)
    omega = omega_sq_oneway_ln(groups)
    pd.DataFrame([
        {
            "codec": codec,
            "pattern": pattern,
            "F": anova["F"],
            "df_between": anova["df_between"],
            "df_within": anova["df_within"],
            "pvalue": anova["pvalue"],
            "omega_sq": omega,
        }
    ]).to_csv(TABLE_DIR / f"tab_5_6b_{token}_welch_anova.csv", index=False)

    gh = games_howell_ln(runs[["format", "ln_time"]].copy(), "format", "ln_time")
    gh.to_csv(TABLE_DIR / f"tab_5_6c_{token}_pairwise_gameshowell.csv", index=False)
    letters = compact_letter_display(gh)
    pd.DataFrame({
        "format": list(letters.keys()),
        "letters": list(letters.values()),
    }).sort_values("format").to_csv(
        TABLE_DIR / f"tab_5_6c_{token}_letters.csv",
        index=False,
    )

    index_map = {fmt: pos for pos, fmt in enumerate(order)}
    matrix = np.ones((len(order), len(order)), dtype=float)
    for row in gh.itertuples(index=False):
        i_pos = index_map[row.i]
        j_pos = index_map[row.j]
        matrix[i_pos, j_pos] = row.ratio
        matrix[j_pos, i_pos] = 1.0 / row.ratio if row.ratio != 0 else np.nan
    plt.figure(figsize=(5.2, 4.6))
    im = plt.imshow(matrix, aspect="auto", interpolation="nearest")
    plt.xticks(range(len(order)), order, rotation=45, ha="right")
    plt.yticks(range(len(order)), order)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title(f"Pairwise time ratios (row / column) — {codec} {pattern}\n<1 means row faster")
    for i in range(len(order)):
        for j in range(len(order)):
            value = matrix[i, j]
            label = "nan" if not np.isfinite(value) else f"{value:.2f}"
            plt.text(j, i, label, ha="center", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_5_6d_{token}_ratio_heatmap.png", dpi=200)
    plt.savefig(FIGURE_DIR / f"fig_5_6d_{token}_ratio_heatmap.pdf")
    plt.close()

# for codec in CODECS:
#     for pattern in PATTERNS:
analyze_latency("gzip", "slice")


# Latency Analysis New

In [8]:
from typing import Iterable

REF_FORMAT = "zarr"                 # reference for ratios
BLOCK_LENGTHS = (4, 6, 8)             # MBB block-length grid (n=50)
B = BOOTSTRAP_DRAWS
BASE_SEED = RNG_SEED + 97 * CODECS.index("gzip") + 13 * PATTERNS.index("slice")


def _mbb_resample(arr: np.ndarray, block_len: int, rng: np.random.Generator) -> np.ndarray:
    """Moving-block bootstrap with circular blocks."""
    n = len(arr)
    if n == 0:
        return arr
    if block_len <= 1:
        return rng.choice(arr, size=n, replace=True)
    ext = np.concatenate([arr, arr[: block_len - 1]])
    out = np.empty(n, dtype=float)
    pos = 0
    while pos < n:
        start = int(rng.integers(0, n))
        blk = ext[start : start + block_len]
        k = min(block_len, n - pos)
        out[pos : pos + k] = blk[:k]
        pos += k
    return out


def mbb_ci_mean_log(ln_values: np.ndarray, draws: int, block_lengths: Iterable[int], seed: int) -> tuple[float, float, float]:
    """GM point and 95% CI via MBB on ln(time); choose widest CI in log-space."""
    ln_values = np.asarray(ln_values, dtype=float)
    ln_values = ln_values[np.isfinite(ln_values)]
    if ln_values.size == 0:
        return np.nan, np.nan, np.nan
    point_ln = float(ln_values.mean())
    rng = np.random.default_rng(seed)
    candidates = []
    for L in block_lengths:
        means = np.empty(draws, dtype=float)
        for b in range(draws):
            boot = _mbb_resample(ln_values, L, rng)
            means[b] = float(np.mean(boot))
        lo, hi = np.quantile(means, [0.025, 0.975])
        candidates.append((float(lo), float(hi)))
    widths = [hi - lo for lo, hi in candidates]
    lo_ln, hi_ln = candidates[int(np.argmax(widths))]
    return float(np.exp(point_ln)), float(np.exp(lo_ln)), float(np.exp(hi_ln))


def mbb_ci_quantile(values: np.ndarray, q: float, draws: int, block_lengths: Iterable[int], seed: int) -> tuple[float, float, float]:
    """q-quantile point and 95% CI via MBB on raw values; choose widest CI."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    point = float(np.quantile(values, q))
    rng = np.random.default_rng(seed + 17)
    candidates = []
    for L in block_lengths:
        est = np.empty(draws, dtype=float)
        for b in range(draws):
            boot = _mbb_resample(values, L, rng)
            est[b] = float(np.quantile(boot, q))
        lo, hi = np.quantile(est, [0.025, 0.975])
        candidates.append((float(lo), float(hi)))
    widths = [hi - lo for lo, hi in candidates]
    lo, hi = candidates[int(np.argmax(widths))]
    return point, lo, hi


def analyze_latency_mbb(codec: str, pattern: str, ref_format: str = REF_FORMAT) -> None:
    runs = load_latency_runs(DATA_DIR, codec, pattern)
    if runs.empty:
        raise ValueError(f"No runs for {codec}/{pattern}.")
    token = f"{pattern}_{codec}"

    # Per-format GM (ms) with dependence-robust CI; p95 (ms) with MBB CI
    summary_rows = []
    for fmt, grp in runs.groupby("format"):
        ln_vals = grp["ln_time"].to_numpy()
        lat_ms = 1000.0 * grp["time_sec"].to_numpy()
        gm_s, gm_lo_s, gm_hi_s = mbb_ci_mean_log(ln_vals, draws=B, block_lengths=BLOCK_LENGTHS, seed=BASE_SEED + hash(fmt) % 10_000)
        gm_ms = 1000.0 * gm_s
        gm_lo_ms = 1000.0 * gm_lo_s
        gm_hi_ms = 1000.0 * gm_hi_s
        p95, p95_lo, p95_hi = mbb_ci_quantile(lat_ms, q=0.95, draws=B, block_lengths=BLOCK_LENGTHS, seed=BASE_SEED + 3_000 + hash(fmt) % 10_000)
        summary_rows.append({
            "format": fmt,
            "n": int(len(grp)),
            "gm_ms": gm_ms,
            "gm_ci_lo_ms": gm_lo_ms,
            "gm_ci_hi_ms": gm_hi_ms,
            "p95_ms": p95,
            "p95_ci_lo_ms": p95_lo,
            "p95_ci_hi_ms": p95_hi,
        })

    summary = pd.DataFrame(summary_rows).sort_values("gm_ms", ascending=True).reset_index(drop=True)

    # Ratios vs reference (point only; descriptive)
    if ref_format in summary["format"].values:
        ref_val = float(summary.loc[summary["format"] == ref_format, "gm_ms"].iloc[0])
        ref_used = ref_format
    else:
        idx_min = int(summary["gm_ms"].idxmin())
        ref_val = float(summary.loc[idx_min, "gm_ms"])
        ref_used = str(summary.loc[idx_min, "format"])
    summary["gm_ratio_vs_ref"] = summary["gm_ms"] / ref_val
    summary["ratio_reference"] = ref_used

    # Save table
    summary.rename(columns={
        "format": "Format",
        "n": "n",
        "gm_ms": "GM_latency_ms",
        "gm_ci_lo_ms": "GM_ci_lo_ms",
        "gm_ci_hi_ms": "GM_ci_hi_ms",
        "p95_ms": "p95_ms",
        "p95_ci_lo_ms": "p95_ci_lo_ms",
        "p95_ci_hi_ms": "p95_ci_hi_ms",
        "gm_ratio_vs_ref": f"GM_ratio_vs_{ref_used}",
        "ratio_reference": "ratio_reference",
    }).to_csv(TABLE_DIR / f"tab_mbb_{token}_latency.csv", index=False)

    # Figure 1: GM ± CI with raw points
    order = summary["format"].tolist()
    x = np.arange(len(order))
    gm = summary["gm_ms"].to_numpy(float)
    gm_lo = summary["gm_ci_lo_ms"].to_numpy(float)
    gm_hi = summary["gm_ci_hi_ms"].to_numpy(float)
    err_low = np.maximum(0.0, gm - gm_lo)
    err_high = np.maximum(0.0, gm_hi - gm)
    yerr = np.vstack([err_low, err_high])

    plt.figure(figsize=(6.6, 3.8))
    plt.errorbar(x, gm, yerr=yerr, fmt="o", capsize=5, elinewidth=1.6)
    # raw points overlay (jittered)
    for xi, fmt in zip(x, order):
        pts = 1000.0 * runs.loc[runs["format"] == fmt, "time_sec"].to_numpy(float)
        jitter = np.random.default_rng(BASE_SEED + 42 + xi).uniform(-0.10, 0.10, size=pts.size)
        plt.plot(np.full_like(pts, xi) + jitter, pts, "o", markersize=2, alpha=0.5)
    plt.xticks(x, order)
    plt.ylabel("Geometric mean latency (ms)")
    plt.title(f"{pattern} — {codec}: GM latency ±95% CI (dependence-robust)")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_gm_ci_ms.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_gm_ci_ms.pdf")
    plt.close()

    # Figure 2: p95 ± CI
    p95 = summary["p95_ms"].to_numpy(float)
    p95_lo = summary["p95_ci_lo_ms"].to_numpy(float)
    p95_hi = summary["p95_ci_hi_ms"].to_numpy(float)
    err95_low = np.maximum(0.0, p95 - p95_lo)
    err95_high = np.maximum(0.0, p95_hi - p95)
    yerr95 = np.vstack([err95_low, err95_high])

    plt.figure(figsize=(6.6, 3.6))
    plt.errorbar(x, p95, yerr=yerr95, fmt="o", capsize=5, elinewidth=1.6)
    plt.xticks(x, order)
    plt.ylabel("p95 latency (ms)")
    plt.title(f"{pattern} — {codec}: p95 latency ±95% CI (block bootstrap)")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_p95_ci_ms.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_p95_ci_ms.pdf")
    plt.close()

    # Figure 3: ratios vs reference (point only; descriptive)
    ratios = summary["gm_ratio_vs_ref"].to_numpy(float)
    plt.figure(figsize=(6.0, 3.2))
    plt.plot(x, ratios, "o")
    plt.axhline(1.0, linestyle="--", linewidth=1)
    plt.xticks(x, order)
    plt.ylabel(f"GM ratio vs {ref_used} (×)")
    plt.title(f"{pattern} — {codec}: Descriptive GM ratios (session-conditional)")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_ratios_point.png", dpi=200)
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_ratios_point.pdf")
    plt.close()

In [ ]:
analyze_latency_mbb("lz4", "slice")
